Instalación y carga de archivos

**Objetivo:** Preparar el entorno de trabajo para el procesamiento de lenguaje natural (NLP), utilizando las siguientes herramientas:

1. **SpaCy:** para el análisis lingüístico, incluyendo lematización y etiquetado de partes de la oración (POS tagging).
2. **NLTK:** para la generación de n-gramas, análisis de colocaciones y cálculo de la Información Mutua Puntual (PMI).


In [2]:
import sys  # Permite acceder a información del sistema, como el ejecutable de Python en uso
import subprocess  # Permite ejecutar comandos del sistema (por ejemplo, instalar paquetes con pip)

# Función para instalar un paquete usando pip
def install(package):
    # Ejecuta el comando: python -m pip install <package>
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import spacy  # Intenta importar la librería SpaCy
except ImportError:
    # Si SpaCy no está instalado, se captura el error y se procede a instalarlo
    print("SpaCy no está instalado. Instalando...")
    install("spacy")  # Llama a la función para instalar el paquete
    import spacy  # Intenta nuevamente importar SpaCy después de la instalación

Cargar modelo en español

In [4]:
try:
    # Intenta cargar el modelo de lenguaje en español de SpaCy
    nlp = spacy.load("es_core_news_sm")
except OSError:
    # Si el modelo no está instalado, se captura el error
    print("Modelo 'es_core_news_sm' no encontrado. Descargando...")

    # Ejecuta el comando para descargar el modelo usando SpaCy
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "es_core_news_sm"])

    # Una vez descargado, intenta cargar nuevamente el modelo
    nlp = spacy.load("es_core_news_sm")

In [5]:
import nltk # Importa la librería principal de NLTK (Procesamiento de Lenguaje Natural)
from nltk.util import ngrams # Importa la función para generar n-gramas (combinaciones de palabras)
from nltk.collocations import BigramCollocationFinder # Importa la clase para encontrar bigramas (pares de palabras frecuentes)
from nltk.metrics import BigramAssocMeasures # Importa medidas estadísticas para evaluar la relación entre palabras
from collections import Counter # Importa Counter para contar frecuencias de elementos

<code> spaCy </code> entiene grámatica

In [7]:
texto_prueba = "Los vuelos fueron retrasados horriblemente y los asientos estaban rotos"
texto_prueba

'Los vuelos fueron retrasados horriblemente y los asientos estaban rotos'

Cuando usamos <code>nlp(texto_prueba)</code>, spaCy procesa el texto completo. La salida no es una lista simple, sino un objeto tipo "Doc".

Este objeto Doc es una estructura avanzada que contiene mucha información lingüística del texto, como:

- **Tokens**: cada palabra o símbolo del texto.
- **Lemas**: la forma base de cada palabra (ej. “corriendo” → “correr”).
- **Etiquetas gramaticales (POS)**: sustantivo, verbo, adjetivo, etc.
- **Dependencias sintácticas**: relaciones entre palabras dentro de la oración.
- **Entidades nombradas (NER)**: nombres de personas, lugares, organizaciones, fechas, etc.


Además, el objeto Doc se puede recorrer fácilmente como si fuera una lista:

In [9]:
documento = nlp(texto_prueba)
documento

Los vuelos fueron retrasados horriblemente y los asientos estaban rotos

In [10]:
# Imprime los encabezados de la tabla con formato alineado
# <20 y <15 indican el ancho de cada columna alineada a la izquierda
print(f"{'Palabra original':<20}|{'Lemma':<15}|{'POS':<15}")

# Imprime una línea separadora de 60 caracteres "_"
print("_"*60)

# Recorre cada token (palabra) dentro del objeto 'documento' (Doc de spaCy)
for token in documento:
    # Imprime:
    # token.text   -> palabra original
    # token.lemma_ -> forma base (lema)
    # token.pos_   -> categoría gramatical (sustantivo, verbo, etc.)
    # Todo alineado en columnas para mejor visualización
    print(f"{token.text:<20}|{token.lemma_:<15}|{token.pos_:<15}")

Palabra original    |Lemma          |POS            
____________________________________________________________
Los                 |el             |DET            
vuelos              |vuelo          |NOUN           
fueron              |ser            |AUX            
retrasados          |retrasar       |VERB           
horriblemente       |horriblemente  |ADV            
y                   |y              |CCONJ          
los                 |el             |DET            
asientos            |asiento        |NOUN           
estaban             |estar          |AUX            
rotos               |roto           |ADJ            


**Explicación de columnas**

1. **Palabra original** → Texto tal cual aparece.
2. **Lemma** → Forma base de la palabra, útil para agrupar palabras con el mismo significado.
3. **POS (Part of Speech)** → Categoría gramatical:
   - **DET** → determinante (*los, el*)
   - **NOUN** → sustantivo (*vuelos, asientos*)
   - **AUX** → verbo auxiliar (*fueron, estaban*)
   - **VERB** → verbo principal (*retrasar*)
   - **ADV** → adverbio (*horriblemente*)
   - **CCONJ** → conjunción (*y*)
   - **ADJ** → adjetivo (*rotos*)

---

**Ejemplo de interpretación de la frase**

*"Los vuelos fueron retrasados horriblemente y los asientos estaban rotos"*

- **“Los vuelos”** → DET + NOUN
- **“fueron retrasados”** → AUX + VERB
- **“horriblemente”** → ADV
- **“y”** → CCONJ
- **“los asientos”** → DET + NOUN
- **“estaban rotos”** → AUX + ADJ

spaCy distingue entre verbos auxiliares, verbos principales y adjetivos, lo que permite un análisis más preciso del texto.

**Usos prácticos**

- Filtrar solo sustantivos o verbos.
- Normalizar texto usando lemas.
- Detectar patrones de lenguaje.
- Preparar datos para análisis de sentimientos.

In [13]:
# Definición de la función que limpia y lematiza un texto
def limpiar_lematiza(texto):
    # Convierte todo el texto a minúsculas y lo procesa con spaCy (Devuelve un objeto Doc)
    documento = nlp(texto.lower())
    
    # Lista vacía donde almacenaremos los tokens limpios y lematizados
    tokens_limpios = []
    
    # Conjunto de etiquetas gramaticales que consideramos importantes:
    # Solo nos interesan sustantivos, adjetivos y verbos
    etiquetas_validas = {"NOUN", "ADJ", "VERB"}
    
    # Conjunto de palabras de negación que queremos mantener
    # Por ejemplo "no", "ni", "nunca" son importantes para análisis de sentimientos
    negaciones = {"no", "ni", "nunca"}

    # Recorremos cada token (palabra) del objeto Doc
    for token in documento:
        # Solo consideramos tokens que contengan letras (ignora números, puntuación, etc.)
        if token.is_alpha:
            # Si la palabra es una negación, la agregamos tal cual
            if token.text in negaciones:
                tokens_limpios.append(token.text)
            # Si la categoría gramatical del token está en nuestras etiquetas válidas,
            # agregamos su lema (forma base) a la lista
            elif token.pos_ in etiquetas_validas:
                tokens_limpios.append(token.lemma_)
    
    # Retorna la lista de tokens limpios y lematizados
    return tokens_limpios

In [14]:
texto_sucio = "¡El servicio no fue bueno! Los asientos estaban súper rotos. 😡😡 "
texto_sucio

'¡El servicio no fue bueno! Los asientos estaban súper rotos. 😡😡 '

In [15]:
limpiar_lematiza(texto_sucio)

['servicio', 'no', 'bueno', 'asiento', 'súper', 'roto']

# Análisis de N-gramas

Los **n-gramas** son secuencias de `n` elementos consecutivos de un texto.  
Se usan en **procesamiento de lenguaje natural (NLP)** para analizar patrones y relaciones entre palabras.

**Tipos de n-gramas y su interpretación**

| Tipo       | Qué captura | Ejemplo con el texto "Hola mundo desde Jupyter" |
|-----------|------------|------------------------------------------------|
| **Unigrama (n=1)** | Palabras individuales; **pierde contexto** | Hola, mundo, desde, Jupyter |
| **Bigram (n=2)** | Captura la **relación inmediata** entre palabras | Hola mundo, mundo desde, desde Jupyter |
| **Trigram (n=3)** | Captura la **idea completa** en secuencias de tres palabras | Hola mundo desde, mundo desde Jupyter |

💡 **Resumen:**  
- Los **unigramas** son simples pero pierden la relación entre palabras.  
- Los **bigramas** muestran cómo una palabra se relaciona con la siguiente.  
- Los **trigramas** permiten entender frases cortas completas o ideas más complejas.

## La maldición de la dimensionalidad en n-gramas

Cuando aumentamos `n` en los n-gramas, el número de combinaciones posibles crece **exponencialmente**. Esto se conoce como la **maldición de la dimensionalidad** y tiene varias consecuencias:

 **Ejemplo**

Texto: `"Hola mundo desde Jupyter"`

- **Unigramas (n=1):** 4 palabras → 4 posibles elementos  
- **Bigramas (n=2):** combinaciones consecutivas → 3 elementos  
- **Trigramas (n=3):** combinaciones consecutivas → 2 elementos  



Ahora imagina un texto largo de **10,000 palabras**:

| Tipo de n-grama | Número aproximado de combinaciones |
|-----------------|----------------------------------|
| Unigramas       | 10,000                          |
| Bigramas        | 10,000² = 100,000,000           |
| Trigramas       | 10,000³ = 1,000,000,000,000     |



**Problemas principales**

1. **Sparsity (escasez de datos)**: muchos n-gramas posibles nunca aparecen en el corpus, por lo que la matriz de ocurrencias queda muy vacía.  
2. **Consumo de memoria**: almacenar todas las combinaciones posibles requiere muchísima memoria.  
3. **Sobreajuste**: modelos que usan n-gramas grandes pueden aprender combinaciones raras que no generalizan bien.

💡 **Resumen:**  
- Los **unigramas** son fáciles de manejar pero pierden contexto.  
- Los **bigramas** y **trigramas** capturan relaciones más complejas, pero crecen **muy rápido** en número y consumen recursos.  
- Es un balance entre **contexto capturado** y **complejidad computacional**.

In [18]:
tokens_base = limpiar_lematiza(texto_sucio)
print(f"Tokens_base: {tokens_base}")

Tokens_base: ['servicio', 'no', 'bueno', 'asiento', 'súper', 'roto']


In [19]:
bigramas = list(ngrams(tokens_base,2))
bigramas

[('servicio', 'no'),
 ('no', 'bueno'),
 ('bueno', 'asiento'),
 ('asiento', 'súper'),
 ('súper', 'roto')]

In [20]:
trigramas = list(ngrams(tokens_base,3))
trigramas

[('servicio', 'no', 'bueno'),
 ('no', 'bueno', 'asiento'),
 ('bueno', 'asiento', 'súper'),
 ('asiento', 'súper', 'roto')]

In [21]:
texto_largo = "El pasajero dijo que el vuelo llegó tarde y la comida estaba fría y sabía mal"
texto_largo 

'El pasajero dijo que el vuelo llegó tarde y la comida estaba fría y sabía mal'

In [22]:
documento = nlp(texto_largo )
documento

El pasajero dijo que el vuelo llegó tarde y la comida estaba fría y sabía mal

In [23]:
# Imprime los encabezados de la tabla con formato alineado
# <20 y <15 indican el ancho de cada columna alineada a la izquierda
print(f"{'Palabra original':<20}|{'Lemma':<15}|{'POS':<15}")

# Imprime una línea separadora de 60 caracteres "_"
print("_"*60)

# Recorre cada token (palabra) dentro del objeto 'documento' (Doc de spaCy)
for token in documento:
    # Imprime:
    # token.text   -> palabra original
    # token.lemma_ -> forma base (lema)
    # token.pos_   -> categoría gramatical (sustantivo, verbo, etc.)
    # Todo alineado en columnas para mejor visualización
    print(f"{token.text:<20}|{token.lemma_:<15}|{token.pos_:<15}")

Palabra original    |Lemma          |POS            
____________________________________________________________
El                  |el             |DET            
pasajero            |pasajero       |NOUN           
dijo                |decir          |VERB           
que                 |que            |SCONJ          
el                  |el             |DET            
vuelo               |vuelo          |NOUN           
llegó               |llegar         |VERB           
tarde               |tarde          |ADV            
y                   |y              |CCONJ          
la                  |el             |DET            
comida              |comida         |NOUN           
estaba              |estar          |AUX            
fría                |frío           |ADJ            
y                   |y              |CCONJ          
sabía               |saber          |VERB           
mal                 |mal            |ADV            


In [24]:
tokens_base = limpiar_lematiza(texto_largo )
print(f"Tokens_base: {tokens_base}")

Tokens_base: ['pasajero', 'decir', 'vuelo', 'llegar', 'comida', 'frío', 'saber']


In [25]:
bigramas = list(ngrams(tokens_base,2))
bigramas

[('pasajero', 'decir'),
 ('decir', 'vuelo'),
 ('vuelo', 'llegar'),
 ('llegar', 'comida'),
 ('comida', 'frío'),
 ('frío', 'saber')]

In [26]:
trigramas = list(ngrams(tokens_base,3))
trigramas

[('pasajero', 'decir', 'vuelo'),
 ('decir', 'vuelo', 'llegar'),
 ('vuelo', 'llegar', 'comida'),
 ('llegar', 'comida', 'frío'),
 ('comida', 'frío', 'saber')]

In [27]:
pentagramas = list(ngrams(tokens_base,5))
pentagramas

[('pasajero', 'decir', 'vuelo', 'llegar', 'comida'),
 ('decir', 'vuelo', 'llegar', 'comida', 'frío'),
 ('vuelo', 'llegar', 'comida', 'frío', 'saber')]

# Pointwise Mutual Information (PMI)

El **PMI** (*Pointwise Mutual Information*) es una medida utilizada en **Procesamiento de Lenguaje Natural (NLP)** para evaluar **cuánto más relacionadas están dos palabras de lo que esperaríamos por azar**.  
Se usa comúnmente para analizar **bigrams** y encontrar palabras que co-ocurren frecuentemente.


**Definición**

Para dos palabras $w_1$ y $w_2$, el PMI se define como:

$$
\text{PMI}(w_1, w_2) =  \frac{P(w_1, w_2)}{P(w_1) \cdot P(w_2)}
$$

Donde:  

- $P(w_1, w_2)$ = probabilidad de que $w_1$ y $w_2$ ocurran **juntas** (como bigrama)  
- $P(w_1), P(w_2)$ = probabilidades de que cada palabra ocurra **independientemente**


**Interpretación de los valores**

| Valor PMI             | Significado |
|----------------------|------------|
| $> 0$                  | Las palabras ocurren juntas **más de lo esperado** (co-ocurrencia positiva) |
| $= 0$                  | Las palabras son **independientes** |
| $< 0$                  | Las palabras ocurren juntas **menos de lo esperado** (co-ocurrencia negativa) |

💡 **Ejemplos intuitivos**:

- `"rey"` y `"reina"` → alto PMI → aparecen juntas con frecuencia  
- `"rey"` y `"zanahoria"` → PMI cercano a $0$ o negativo → raramente aparecen juntas  


In [31]:
# Frecuencia y PMI
resenas = [
    "La atención médica fue excelente y el doctor muy amable.",
    "El seguro médico no cubrió la operación",
    "La atención en urgencias fue lenta, pero el doctor fue bueno",
    "Necesito mi historial médico urgente",
    "La atención médica fue lenta",
    "El historial médico no apareció"
]
resenas

['La atención médica fue excelente y el doctor muy amable.',
 'El seguro médico no cubrió la operación',
 'La atención en urgencias fue lenta, pero el doctor fue bueno',
 'Necesito mi historial médico urgente',
 'La atención médica fue lenta',
 'El historial médico no apareció']

In [32]:
tokens_todos = []  # Lista vacía donde se almacenarán todos los tokens procesados de todas las reseñas

for resena in resenas:  # Itera sobre cada reseña en la lista 'resenas'
    tokens_procesados = limpiar_lematiza(resena)  # Llama a la función 'limpiar_lematiza' para limpiar y lematizar la reseña
    print(f"tokens_procesados: {tokens_procesados}")  # Muestra los tokens procesados de la reseña actual
    tokens_todos.extend(tokens_procesados)  # Agrega los tokens procesados a la lista general 'tokens_todos'

print("Tokens totales")  # Mensaje indicando que se mostrarán todos los tokens
print(tokens_todos)  # Imprime la lista completa de tokens de todas las reseñas

tokens_procesados: ['atención', 'médico', 'excelente', 'doctor', 'amable']
tokens_procesados: ['seguro', 'médico', 'no', 'cubrir', 'operación']
tokens_procesados: ['atención', 'urgencia', 'lento', 'doctor', 'bueno']
tokens_procesados: ['necesitar', 'historial', 'médico', 'urgente']
tokens_procesados: ['atención', 'médico', 'lento']
tokens_procesados: ['historial', 'médico', 'no', 'aparecer']
Tokens totales
['atención', 'médico', 'excelente', 'doctor', 'amable', 'seguro', 'médico', 'no', 'cubrir', 'operación', 'atención', 'urgencia', 'lento', 'doctor', 'bueno', 'necesitar', 'historial', 'médico', 'urgente', 'atención', 'médico', 'lento', 'historial', 'médico', 'no', 'aparecer']


In [33]:
buscado = BigramCollocationFinder.from_words(tokens_todos)  
# Crea un objeto BigramCollocationFinder usando todos los tokens de todas las reseñas
# Este objeto permite buscar bigramas (pares de palabras consecutivas) y analizar su frecuencia o asociación

buscado.apply_freq_filter(2)  
# Filtra los bigramas que aparecen menos de 2 veces en el corpus
# Esto ayuda a ignorar combinaciones raras que no son relevantes

metricas = BigramAssocMeasures()  
# Crea un objeto con métricas estadísticas para evaluar la fuerza de asociación de los bigramas
# Entre estas métricas están PMI (Pointwise Mutual Information) y frecuencia cruda

print("\nTop bigramas por frecuencia")  
# Mensaje indicando que se mostrarán los bigramas más frecuentes
print(buscado.score_ngrams(metricas.raw_freq)[:5])  
# Devuelve los 5 bigramas más frecuentes según la frecuencia cruda

print("\nTop bigramas por PMI")  
# Mensaje indicando que se mostrarán los bigramas con mayor PMI
print(buscado.score_ngrams(metricas.pmi)[:5])  
# Devuelve los 5 bigramas con mayor asociación según PMI (indicando combinaciones más informativas)


Top bigramas por frecuencia
[(('atención', 'médico'), 0.07692307692307693), (('historial', 'médico'), 0.07692307692307693), (('médico', 'no'), 0.07692307692307693)]

Top bigramas por PMI
[(('historial', 'médico'), 2.3785116232537304), (('médico', 'no'), 2.3785116232537304), (('atención', 'médico'), 1.793549122532574)]
